Теория
Случайный лес (Random Forest) — это ансамблевый метод машинного обучения, основанный на комбинации нескольких деревьев решений. Он используется как для задач классификации, так и для регрессии и помогает снизить переобучение, улучшая точность модели.
Основные принципы работы:

Ансамбль деревьев решений — случайный лес строит множество деревьев и объединяет их предсказания.
Бэггинг (Bootstrap Aggregating) — каждое дерево обучается на случайной подвыборке данных.
Случайный выбор признаков — при построении каждого дерева выбирается случайное подмножество признаков, что делает модель более устойчивой к шуму и переобучению.

Преимущества:

Уменьшает переобучение по сравнению с одиночным деревом решений.
Хорошо работает с разнородными данными.
Поддерживает обработку как числовых, так и категориальных признаков.
Недостатки:

Модель сложнее в интерпретации по сравнению с одним деревом.
Может требовать значительных вычислительных ресурсов при большом количестве деревьев.
Ключевым недостатком деревьев решений является их «склонность» к переобучению, а также тот факт, что даже при небольшом изменении обучающей выборки дерево может значительно измениться. Однако объединение деревьев в ансамбли или композиции на практике дает очень хорошие результаты.

Ансамбли — это методы, сочетающие в себе несколько алгоритмов машинного обучения для повышения качества работы модели.
В случае задачи регрессии при использовании композиции a(x) из N  базовых алгоритмов b^n(x) ответом будет считаться среднее значение ответа каждого алгоритма:

$a(x) = \frac{1}{N}\sum_{n=1}^{N}b_{n}(x)$

В задачах классификации, соответственно, знак полученного усредненного ответа или (что аналогично) класс определяется путем голосования: объект относится к классу, за который «проголосовало» наибольшее число базовых алгоритмов.
Случайный лес формируют из деревьев решения. «Случайными» леса названы из-за того, что в процесс построения деревьев, из которых они состоят, внесен элемент случайности для обеспечения уникальности каждого из деревьев. Случайность заключается в обучении базовых алгоритмов на разных подвыборках обучающей выборки.

Один из способов построения случайных подвыборок — бутстрап (bootstrap).

Bootstrap — это метод выборки с возвращением, который используется для создания нескольких подвыборок из исходных данных.

Этот метод заключается в получении из выборки длины l нескольких разных выборок той же длины l за счет копирования элементов. Для получения бутстрап-выборки из исходной выборки l раз выбирается случайный элемент, причем каждый раз новый элемент выбирается из всей выборки. Таким образом, в полученной в конечном итоге бутстрап-выборке некоторые элементы исходной выборки будут встречаться несколько раз, а некоторые (примерно 37% выборки) будут вовсе отсутствовать, и при повторении N раз мы получим N разных выборок длиной l.

Например, если у нас есть исходная выборка вида [a, b, c, d, e], возможными бутстрап-выборками могут быть [a, b, a, c, b] или [b, e, e, d, b] и пр.
При построении случайного леса вначале создается набор бутстрап-выборок под каждое будущее дерево в «лесу». Для уменьшения корреляции базовых алгоритмов случайным делается и процесс построения каждого дерева.

В стандартном методе построения деревьев в каждом узле выбирается j-й признак и порог t, с которым сравнивается его значение, и потом эти значения оптимизируются с помощью функции ошибки.
А в методе случайного леса в каждой вершине j-й признак выбирается не из всего пространства признаков, а из его случайного подмножества размера m, которое каждый раз выбирается заново (в этом отличие от метода случайных подпространств, где подпространство выбирается единожды и используется для построения всего дерева).
В задачах классификации рекомендуется брать $m = \sqrt{d}$, где  d— общее число признаков, и строить дерево до тех пор, пока в каждом листе не останется по одному объекту, а в задаче регрессии — принимать  $m = d/3$ и строить дерево, пока в листьях не останется по пять объектов.

Далее построенные деревья объединяются в композицию, и при предсказаниях с его помощью используется усредненный ответ на каждом дереве.

In [3]:
'''Реализация случайного леса в scikit-learn
Рассмотрим использование случайного леса для классификации датасета ирисов Фишера:'''

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Загрузка данных
iris = load_iris()
X, y = iris.data, iris.target

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создание и обучение модели
clf = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42)
clf.fit(X_train, y_train)

# Предсказание и оценка качества модели
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность модели: {accuracy:.2f}')

Точность модели: 1.00


In [ ]:
'''Гиперпараметры у деревьев и леса
Конструктор случайного леса для регрессии (посмотрим для разнообразия на регрессионный алгоритм, для классификации — аналогичные):
'''
class sklearn.ensemble.RandomForestRegressor(
    n_estimators -- число деревьев в "лесу" (по дефолту - 10)
    criterion -- функция, которая измеряет качество разбиения ветки дерева (по дефолту -- "mse" , так же можно выбрать "mae")
    max_features -- число признаков, по которым ищется разбиение. Вы можете указать конкретное число или процент признаков, либо выбрать из доступных значений: "auto" (все признаки), "sqrt", "log2". По дефолту стоит "auto".
    max_depth -- максимальная глубина дерева  (по дефолту глубина не ограничена)
    min_samples_split -- минимальное количество объектов, необходимое для разделения внутреннего узла. Можно задать числом или процентом от общего числа объектов (по дефолту -- 2)
    min_samples_leaf -- минимальное число объектов в листе. Можно задать числом или процентом от общего числа объектов (по дефолту -- 1)
    min_weight_fraction_leaf -- минимальная взвешенная доля от общей суммы весов (всех входных объектов) должна быть в листе (по дефолту имеют одинаковый вес)
    max_leaf_nodes -- максимальное количество листьев (по дефолту нет ограничения)
    min_impurity_split -- порог для остановки наращивания дерева (по дефолту 1е-7)
    bootstrap -- применять ли бустрэп для построения дерева (по дефолту True)
    oob_score -- использовать ли out-of-bag объекты для оценки R^2 (по дефолту False)
    n_jobs -- количество ядер для построения модели и предсказаний (по дефолту 1, если поставить -1, то будут использоваться все ядра)
    random_state -- начальное значение для генерации случайных чисел (по дефолту его нет, если хотите воспроизводимые результаты, то нужно указать любое число типа int
    verbose -- вывод логов по построению деревьев (по дефолту 0)
    warm_start -- использует уже натренированую модель и добавляет деревьев в ансамбль (по дефолту False)
)

Все эти гиперпараметры вы можете задавать самостоятельно и, сравнивая результаты, подбирать наилучшие. В следующих модулях будем изучать, как это делается готовыми методами, а пока можно вручную попробовать изменять гиперпараметры при создании алгоритма.

Случайный лес позволяет получать точные и устойчивые результаты. Он широко применяется в задачах прогнозирования, анализа данных и распознавания образов. Для улучшения модели можно настраивать гиперпараметры, такие как количество деревьев, глубина деревьев и количество признаков при разбиении.